# Ad Click Prediction
Predict whether a user will click on an online advertisement.

## Project Overview
Binary classification to predict ad clicks from user demographics and browsing behavior. A core digital marketing ML task.

## Learning Objectives
- Build a CTR prediction model
- Work with user behavior features
- Evaluate binary classifiers with ROC-AUC

## Problem Statement
Given user demographics and browsing data, predict whether they will click on an ad (binary classification).

## Why This Project Matters
CTR prediction powers the multi-billion dollar digital advertising industry. Better models mean less wasted ad spend and better user experience.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Kaggle: gabrielsantello/advertisement-click-on-ad |
| **Target** | Clicked on Ad (binary) |
| **Rows** | ~1,000 |
| **Features** | Age, area income, daily internet usage, time on site, gender, etc. |

## Dataset Source & License
Kaggle advertising dataset for learning. Public domain.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
# Target column will be auto-detected

## Dataset Loading

In [4]:
import kagglehub
path = kagglehub.dataset_download('gabrielsantello/advertisement-click-on-ad')
import glob
csvs = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
print('Files:', csvs)
df = pd.read_csv(csvs[0])
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
df.head()

Files: ['C:\\Users\\ahmad\\.cache\\kagglehub\\datasets\\gabrielsantello\\advertisement-click-on-ad\\versions\\1\\advertising.csv']
Shape: (1000, 10)


,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Ad Topic Line,City,Male,Country,Timestamp,Clicked on Ad
0,68.95,35,61833.90,256.09,Cloned 5thgeneration orchestration,Wrightburgh,0,Tunisia,2016-03-27 00:53:11,0
1,80.23,31,68441.85,193.77,Monitored national standardization,West Jodi,1,Nauru,2016-04-04 01:39:02,0
2,69.47,26,59785.94,236.50,Organic bottom-line service-desk,Davidton,0,San Marino,2016-03-13 20:35:42,0
3,74.15,29,54806.18,245.89,Triple-buffered reciprocal time-frame,West Terrifurt,1,Italy,2016-01-10 02:31:19,0
4,68.37,35,73889.99,225.58,Robust logistical utilization,South Manuel,0,Iceland,2016-06-03 03:36:18,0


## Data Validation

In [5]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

# Find target
click_cols = [c for c in df.columns if 'click' in c.lower()]
TARGET = click_cols[0] if click_cols else df.columns[-1]
print(f'\nTarget: {TARGET}')
print(df[TARGET].value_counts())

Missing values:
Daily Time Spent on Site    0
Age                         0
Area Income                 0
Daily Internet Usage        0
Ad Topic Line               0
City                        0
Male                        0
Country                     0
Timestamp                   0
Clicked on Ad               0
dtype: int64

Duplicates: 0

Target: Clicked on Ad
Clicked on Ad
0    500
1    500
Name: count, dtype: int64


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#3498db','#e74c3c'], edgecolor='black')
axes[0,0].set_title('Click Distribution')

num_cols = df.select_dtypes('number').columns.tolist()
num_cols = [c for c in num_cols if c != TARGET][:3]
for i, col in enumerate(num_cols):
    ax = axes[(i+1)//2, (i+1)%2]
    for label in [0, 1]:
        subset = df[df[TARGET] == label]
        ax.hist(subset[col].dropna(), bins=30, alpha=0.5, label=f'{TARGET}={label}', edgecolor='black')
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].value_counts())
print(f'\nBalance: {df[TARGET].mean():.2%} positive')

Clicked on Ad
0    500
1    500
Name: count, dtype: int64

Balance: 50.00% positive


## Train/Test Split

In [8]:
# Drop non-feature columns
drop_cols = [c for c in df.columns if any(kw in c.lower() for kw in ['timestamp','ad topic','city','country'])]
df = df.drop(columns=drop_cols, errors='ignore')

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c] = df[c].astype('category').cat.codes

df = df.fillna(df.median(numeric_only=True))

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (800, 5), Test: (200, 5)


## Preprocessing
Dropped timestamp and text-heavy columns. Encoded categoricals as integer codes.

## Feature Engineering
Using demographic and browsing behavior features directly.

## Baseline Model

In [9]:
bl = LogisticRegression(max_iter=1000, random_state=SEED)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}')
try:
    bl_proba = bl.predict_proba(X_test)[:, 1]
    print(f'  ROC-AUC={roc_auc_score(y_test, bl_proba):.4f}')
except: pass

Baseline LR: Acc=0.9800  F1=0.9798
  ROC-AUC=0.9903


## LazyPredict Benchmark

In [10]:
# --- LazyPredict Benchmark ---
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  \
Model                                                                           
CalibratedClassifierCV            0.980              0.980  0.98980  0.979998   
LinearSVC                         0.980              0.980  0.98960  0.979998   
LogisticRegression                0.980              0.980  0.99020  0.979998   
GaussianNB                        0.975              0.975  0.99240  0.974999   
NuSVC                             0.975              0.975  0.99020  0.974994   
NearestCentroid                   0.975              0.975  0.99160  0.974984   
PassiveAggressiveClassifier       0.975              0.975  0.99120  0.974999   
LinearDiscriminantAnalysis        0.970              0.970  0.99130  0.969988   
RandomForestClassifier            0.970              0.970  0.99115  0.970000   
QuadraticDiscriminantAnalysis     0.970              0.970  0.98950  0.970000   
KNeighborsClassifier        

## FLAML AutoML

In [11]:
# --- FLAML AutoML ---
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Accuracy={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
    if hasattr(automl, 'predict_proba'):
        try:
            flaml_proba = automl.predict_proba(X_test)
            if flaml_proba.shape[1] == 2:
                print(f"  ROC-AUC={roc_auc_score(y_test, flaml_proba[:, 1]):.4f}")
            else:
                print(f"  ROC-AUC={roc_auc_score(y_test, flaml_proba, multi_class='ovr', average='weighted'):.4f}")
        except: pass
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
# --- Boosting Models ---
models = {}
# CatBoost
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

# LightGBM
lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

# XGBoost
xgb = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

CatBoost: Acc=0.9750  F1=0.9750
LightGBM: Acc=0.9550  F1=0.9550
XGBoost: Acc=0.9600  F1=0.9600


## Top 3 Model Selection

In [13]:
# --- Top 3 Selection ---
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }

results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

          Accuracy        F1  Precision  Recall
CatBoost     0.975  0.974999   0.975048   0.975
XGBoost      0.960  0.959996   0.960184   0.960
LightGBM     0.955  0.954999   0.955046   0.955

Top 3: ['CatBoost', 'XGBoost', 'LightGBM']


## Final Evaluation of Top 3

In [14]:
# --- Final Evaluation of Top 3 ---
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

CatBoost: Acc=0.9750  F1=0.9750
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       100
           1       0.98      0.97      0.97       100

    accuracy                           0.97       200
   macro avg       0.98      0.97      0.97       200
weighted avg       0.98      0.97      0.97       200

XGBoost: Acc=0.9600  F1=0.9600
              precision    recall  f1-score   support

           0       0.97      0.95      0.96       100
           1       0.95      0.97      0.96       100

    accuracy                           0.96       200
   macro avg       0.96      0.96      0.96       200
weighted avg       0.96      0.96      0.96       200



LightGBM: Acc=0.9550  F1=0.9550
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       100
           1       0.96      0.95      0.95       100

    accuracy                           0.95       200
   macro avg       0.96      0.95      0.95       200
weighted avg       0.96      0.95      0.95       200



## Error Analysis

In [15]:
# --- Error Analysis ---
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

# Misclassification analysis
wrong = X_test[preds != y_test]
right = X_test[preds == y_test]
print(f'Correct: {len(right)}, Wrong: {len(wrong)}, Error rate: {len(wrong)/len(y_test):.4f}')

# Feature importance if available
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

Correct: 195, Wrong: 5, Error rate: 0.0250


## Interpretation & Business Insight
Age, area income, and daily internet usage are strong predictors. Higher income users with less internet usage tend to click more — targeting insights for advertisers.

## Limitations
- Small dataset (~1K rows)
- Synthetic/simplified — real CTR data has billions of rows
- Missing contextual features (ad type, placement)

## How to Improve
- Add time-of-day features
- Include ad content features
- Use interaction features (age × income)

## Production Considerations
- Real-time scoring needed for ad bidding
- Model must handle cold-start users
- A/B testing framework essential

## Common Mistakes
- Using city/country directly (too many levels)
- Not considering class balance
- Ignoring feature importance for targeting decisions

## Mini Challenge
1. Engineer age buckets and income quintiles
2. Try time-on-site × pages-per-visit interaction
3. Plot precision-recall curve

## Final Summary
Built ad click prediction models. Even simple demographics predict clicks well on this dataset.

In [16]:
# --- Save Metrics ---
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "CatBoost": {
    "Accuracy": 0.975,
    "F1": 0.975,
    "Precision": 0.975,
    "Recall": 0.975
  },
  "XGBoost": {
    "Accuracy": 0.96,
    "F1": 0.96,
    "Precision": 0.9602,
    "Recall": 0.96
  },
  "LightGBM": {
    "Accuracy": 0.955,
    "F1": 0.955,
    "Precision": 0.955,
    "Recall": 0.955
  },
  "best_model": "CatBoost"
}
